### Phase 2 & 3: Experiment Tracking, Training, and Model Registration

This notebook trains multiple classification models using the Breast Cancer dataset prepared in Phase 1.

Steps:
1. Load the cleaned Delta table
2. Split data into train/test sets
3. Enable MLflow autologging
4. Run at least three experiments
5. Compare accuracy, F1 score, and ROC AUC
6. Select the best model
7. Prepare the model for registration

In [0]:
# Load dataset prepared in Phase 1

current_catalog = spark.sql("SELECT current_catalog()").collect()[0][0]
print("Current catalog:", current_catalog)

catalog = current_catalog
schema = "default"
table_name = "breast_cancer"

full_table_name = f"{catalog}.{schema}.{table_name}"

spark_df = spark.table(full_table_name)

display(spark_df)
spark_df.printSchema()

In [0]:
# Convert to Pandas & split Features/Label

import pandas as pd

pdf = spark_df.toPandas()

X = pdf.drop(columns=["target"])
y = pdf["target"]

print("X shape:", X.shape)
print("y shape:", y.shape)

display(pdf.head())


In [0]:
# Train / test split

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train rows:", X_train.shape[0])
print("Test rows:", X_test.shape[0])


In [0]:
import mlflow
import mlflow.sklearn

experiment_name = "/Users/omar2004@gmail.com/mlops-capstone-experiments"

mlflow.set_experiment(experiment_name)

mlflow.sklearn.autolog(
    log_input_examples=True,
    log_model_signatures=True,
    silent=True
)

print("MLflow experiment:", experiment_name)

In [0]:
# Run 3 experiments (core requirement)

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

configs = [
    {"n_estimators": 50, "max_depth": 3},
    {"n_estimators": 100, "max_depth": 5},
    {"n_estimators": 200, "max_depth": 8},
]

results = []

best_run_id = None
best_auc = -1
best_model = None

for cfg in configs:
    with mlflow.start_run(run_name=f"rf_{cfg['n_estimators']}_{cfg['max_depth']}"):
        model = Pipeline([
            ("scaler", StandardScaler()),
            ("rf", RandomForestClassifier(
                n_estimators=cfg["n_estimators"],
                max_depth=cfg["max_depth"],
                random_state=42
            ))
        ])

        model.fit(X_train, y_train)

        preds = model.predict(X_test)
        probs = model.predict_proba(X_test)[:, 1]

        acc = accuracy_score(y_test, preds)
        f1 = f1_score(y_test, preds)
        auc = roc_auc_score(y_test, probs)

        mlflow.log_params(cfg)
        mlflow.log_metrics({
            "accuracy": acc,
            "f1": f1,
            "auc": auc
        })

        results.append({
            "run_id": mlflow.active_run().info.run_id,
            "accuracy": acc,
            "f1": f1,
            "auc": auc,
            **cfg
        })

        if auc > best_auc:
            best_auc = auc
            best_run_id = mlflow.active_run().info.run_id
            best_model = model

results_df = pd.DataFrame(results)
display(results_df)

print("Best run ID:", best_run_id)
print("Best AUC:", best_auc)


## Model Selection Rationale

Three Random Forest models with different hyperparameters were trained and evaluated.

The primary selection metric was **ROC AUC**, as it provides a threshold‑independent measure of classification performance.

The model with the highest ROC AUC was selected as the best model. Accuracy and F1 score were also reviewed to ensure balanced performance.